QC, normalization and Tangram mapping (cluster mode) for one sample, set via `sample`.
- Output: `adatas/per_sample/<sample>.h5ad` with cell type probabilities per spot, plus spatial probability maps.

In [ ]:
sample = "739"

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
import tangram as tg
from sklearn.preprocessing import minmax_scale
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/FOXF2/Visium")
spaceranger = Path("/dss/dssfs03/pn52re/pn52re-dss-0000/2021-FOXF2/visium/spaceranger")
condition = {"733": "KO", "734": "KO", "738": "KO", "740": "KO", "735": "WT", "737": "WT", "739": "WT", "741": "WT"}[sample]
(base / "adatas" / "per_sample").mkdir(parents=True, exist_ok=True)
(base / "plots").mkdir(exist_ok=True)

In [ ]:
adata = sq.read.visium(spaceranger / f"sample{sample}" / "outs")
adata.var_names_make_unique()
adata.var["mt"] = adata.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
sc.pl.spatial(adata, color=["total_counts", "n_genes_by_counts", "pct_counts_mt"], bw=True)

In [ ]:
sc.pp.filter_cells(adata, min_counts=500)
sc.pp.filter_genes(adata, min_cells=0)
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["normalized"] = adata.X.copy()
adata.obs["sample"] = sample
adata.obs["condition"] = condition
adata

Tangram, cluster mode: top 125 marker genes per reference cell type as training genes

In [ ]:
ad_sc = sc.read_h5ad(base / "adatas" / "20220823_saunders_annotated.h5ad")
markers = np.unique(pd.DataFrame(ad_sc.uns["rank_genes_groups"]["names"]).iloc[:125].values).tolist()
ad_sp = adata.copy()
tg.pp_adatas(ad_sc, ad_sp, genes=markers)
ad_map = tg.map_cells_to_space(ad_sc, ad_sp, mode="clusters", cluster_label="clusters")
prob = pd.DataFrame(ad_map.X.T, index=ad_map.var_names, columns="Prob " + ad_map.obs["clusters"].astype(str))
adata.obs = adata.obs.join(prob)
assert adata.obs[prob.columns].notna().all().all()

In [ ]:
ad_ge = tg.project_genes(ad_map, ad_sc, cluster_label="clusters")
genes = tg.compare_spatial_geneexp(ad_ge, ad_sp, ad_sc)
tg.plot_training_scores(ad_map, bins=20, alpha=0.5)
scatterplot = sns.scatterplot
sns.scatterplot = lambda x, y, **kwargs: scatterplot(x=x, y=y, **kwargs)  # tangram plot_auc passes x, y positionally
tg.plot_auc(genes)

In [ ]:
scaled = adata.copy()
scaled.obs[prob.columns] = scaled.obs[prob.columns].apply(minmax_scale)
sc.pl.spatial(scaled, color=prob.columns, frameon=False, size=1.55, bw=True, alpha_img=0, cmap="inferno", show=False)
for ext in ("png", "svg"):
    plt.savefig(base / "plots" / f"tangram_probability_scaled_{sample}.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
adata.write_h5ad(base / "adatas" / "per_sample" / f"{sample}.h5ad", compression="gzip")